In [1]:
%%javascript

IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}
// Trying to fix Mol viewer inside accordion box,
// padding 15px is the default, reducing it did not help
var styles = `
    .p-Collapse-contents { 
        padding: 15px;
    }
`
var styleSheet = document.createElement("style")
styleSheet.innerText = styles

document.head.appendChild(styleSheet)

document.title = 'AiiDAlab ATMOSPEC app'
if (document.getElementById('appmode-busy')) {
    window.onbeforeunload = function() {return}
}

<IPython.core.display.Javascript object>

In [2]:
%%html

<style>
  .output_subarea {
    max-width: none !important;
  }
</style>

In [3]:
import ipywidgets as ipw

from aiida import load_profile

profile = load_profile()
loading_message = ipw.HTML(
    value=f"Loaded AiiDA profile {profile.name!r}<br>Hold on to your hats, ATMOSPEC will be here shortly 🚀"
)
display(loading_message)

HTML(value="Loaded AiiDA profile 'default'<br>Hold on to your hats, ATMOSPEC will be here shortly 🚀")

In [4]:
# Activate Bokeh

# https://docs.bokeh.org/en/latest/docs/user_guide/jupyter.html
# https://github.com/bokeh/bokeh/blob/branch-3.0/examples/howto/server_embed/notebook_embed.ipynb
from bokeh.io import output_notebook

# https://docs.bokeh.org/en/latest/docs/reference/io.html#bokeh.io.output_notebook
output_notebook(hide_banner=True, load_timeout=5000, verbose=True)

In [5]:
import os
from importlib.resources import files
from pathlib import Path

# External imports
from jinja2 import Environment

from aiida.orm import StructureData, TrajectoryData, load_node
from aiidalab_widgets_base import (
    StructureBrowserWidget,
    StructureUploadWidget,
    WizardAppWidget,
)
from aiidalab_widgets_base.bug_report import (
    install_create_github_issue_exception_handler,
)

<IPython.core.display.Javascript object>

In [6]:
from aiidalab_ispg import __version__
from aiidalab_ispg.app import (
    ConformerSmilesWidget,
    ISPGWorkChainSelector,
    StructureSelectionStep,
    SubmitAtmospecAppWorkChainStep,
    TrajectoryDataViewer,
    ViewAtmospecAppWorkChainStatusAndResultsStep,
    ViewSpectrumStep,
    static,
)
from aiidalab_ispg.app.widgets import TrajectoryManagerWidget

WORKCHAIN_LABEL = "ATMOSPEC workflow"

structure_manager_widget = TrajectoryManagerWidget(
    importers=[
        ConformerSmilesWidget(title="SMILES"),
        StructureUploadWidget(title="Upload file", allow_trajectories=True),
        StructureBrowserWidget(
            title="AiiDA database", query_types=(StructureData, TrajectoryData)
        ),
    ],
    node_class="TrajectoryData",
    viewer=TrajectoryDataViewer(),
    storable=True,
)

structure_selection_description = ipw.Label(
    'Select a structure from one of the following sources and then click "Confirm" to go to the next step. '
)

structure_selection_step = StructureSelectionStep(
    manager=structure_manager_widget, description=structure_selection_description
)
structure_selection_step.auto_advance = True

submit_atmospec_work_chain_step = SubmitAtmospecAppWorkChainStep()
submit_atmospec_work_chain_step.auto_advance = True

view_atmospec_work_chain_status_and_results_step = (
    ViewAtmospecAppWorkChainStatusAndResultsStep()
)
view_atmospec_work_chain_status_and_results_step.auto_advance = True

view_spectrum_step = ViewSpectrumStep()

# Link the application steps
ipw.dlink(
    (structure_selection_step, "confirmed_structure"),
    (submit_atmospec_work_chain_step, "input_structure"),
)
ipw.dlink(
    (submit_atmospec_work_chain_step, "process"),
    (view_atmospec_work_chain_status_and_results_step, "process_uuid"),
    transform=lambda node: node.uuid if node is not None else None,
)
ipw.dlink(
    (view_atmospec_work_chain_status_and_results_step, "process_uuid"),
    (view_spectrum_step, "process_uuid"),
)

# Add the application steps to the application
app = WizardAppWidget(
    steps=[
        ("Select structure", structure_selection_step),
        ("Submit workflow", submit_atmospec_work_chain_step),
        ("Status & Detailed outputs", view_atmospec_work_chain_status_and_results_step),
        ("UV/VIS Spectrum", view_spectrum_step),
    ]
)


# Reset all subsequent steps in case that a new structure is selected
def _observe_structure_selection(change):
    with structure_selection_step.hold_sync():
        if (
            structure_selection_step.confirmed_structure is not None
            and structure_selection_step.confirmed_structure != change["new"]
        ):
            app.reset()


structure_selection_step.observe(_observe_structure_selection, "structure")

# Add process selection header
work_chain_selector = ISPGWorkChainSelector(
    process_label=WORKCHAIN_LABEL, layout=ipw.Layout(width="auto")
)


def _observe_process_selection(change):
    if change["old"] == change["new"]:
        return
    pk = change["new"]
    if pk is None:
        app.reset()
        app.selected_index = 0
        return

    process = load_node(pk)
    with structure_manager_widget.hold_sync():
        with structure_selection_step.hold_sync():
            if process.is_finished_ok:
                app.selected_index = 3
            else:
                app.selected_index = 2
            structure_manager_widget.input_structure = process.inputs.structure
            structure_selection_step.structure = process.inputs.structure
            structure_selection_step.confirmed_structure = process.inputs.structure
            submit_atmospec_work_chain_step.process = process


work_chain_selector.observe(_observe_process_selection, "value")
# Make sure the Process list in WorkChainSelector is updated after we submit a process
ipw.dlink(
    (submit_atmospec_work_chain_step, "process"),
    (work_chain_selector, "value"),
    transform=lambda node: None if node is None else node.pk,
)
# Make sure the WorkChainSelector is updated when the process is finished
view_atmospec_work_chain_status_and_results_step.process_monitor.on_sealed.append(
    work_chain_selector.refresh_work_chains
)

# Let's render!
env = Environment()
template = files(static).joinpath("welcome.jinja").read_text()
style = files(static).joinpath("style.css").read_text()
# NOTE: The walk_up parameter was only added in Python 3.12,
# so we're using os.path.relpath
# path_to_static = files(static).relative_to(Path.cwd(), walk_up=True)
path_to_static = os.path.relpath(files(static), Path.cwd())
welcome_message = ipw.HTML(
    env.from_string(template).render(style=style, path_to_static=path_to_static)
)

footer = ipw.VBox(
    children=[
        ipw.HTML(
            "Funded by ERC grant 803718 SINDAM and EPSRC grant EP/V026690/1 UPDICE."
        ),
        ipw.HTML(
            f"© 2024 ISPG team (University of Bristol)&#8195Version: {__version__}"
        ),
    ],
    # layout=ipw.Layout(justify_content="space-between"),
)

app_with_work_chain_selector = ipw.VBox(children=[work_chain_selector, app])

error_handler_output = ipw.Output()
install_create_github_issue_exception_handler(
    error_handler_output,
    url="https://github.com/ispg-group/aiidalab-ispg/issues/new",
    labels=("bug", "automated-report"),
)
loading_message.layout.display = "none"
display(welcome_message, error_handler_output, app_with_work_chain_selector, footer)

HTML(value='<head>\n  <style>\n    :root {\n    --lab-blue: #2097F3;\n    --lab-background: #d3ecff;\n}\n\nh3 …

Output()

/home/jovyan/.local/lib/python3.9/site-packages/aiidalab_widgets_base/viewers.py:727: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  self.cell_spacegroup.value = f"Spacegroup: {symmetry_dataset['international']} (No.{symmetry_dataset['number']})"


uuid: dbf607bd-a020-4a75-afc0-00090087684f (pk: 552) (aiidalab_ispg.workflows.atmospec.AtmospecWorkChain)


/opt/conda/lib/python3.9/site-packages/aiida/orm/querybuilder.py:887: AiidaDeprecationWarning: `QueryBuilder.set_debug` is deprecated. Configure the log level of the AiiDA logger instead. (this will be removed in v3)
  warn_deprecation(
